# 🧪 Practice Lab: Math Foundations for GenAI

**Netsetos GenAI Course** | Module 0 · Lesson 0.2 | ⏱ ~60-90 min

**Prerequisites:** Basic Python + NumPy (Google Colab has both pre-installed)

### 🎯 You will:
1. Build a similarity search engine with cosine similarity
2. Explore how temperature controls softmax distributions
3. Implement self-attention (Q/K/V) from scratch
4. Calculate perplexity to evaluate language models
5. Train a 3-layer network with GELU activation via backprop
6. Build mini Word2Vec and test word analogies

---

## Exercise 1 (Easy): Similarity Search Engine

**📖 Concept:** Cosine similarity = `dot(a,b) / (‖a‖ × ‖b‖)`. Measures angle between vectors — magnitude-invariant. Score >0.85 = relevant. Every vector DB (ChromaDB, Pinecone) uses this as default.

**📝 Task:**
1. Create 8 document embeddings (4 tech, 4 food) as NumPy arrays
2. Create a query vector for "how to make dosa"
3. Compute cosine similarity between the query and every document
4. Print all scores sorted — food docs should rank highest

**📤 Expected Output:**
```
  0.996  How to make dosa at home
  0.992  Best biryani recipe
  0.987  Indian street food guide
  0.310  Deploying ML on GCP
  ...  (food docs rank highest)
```

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np

def cosine_similarity(a, b):
    """Cosine similarity: dot(a,b) / (norm(a) * norm(b))"""
    # TODO: implement
    pass

# 8 documents: 4 tech, 4 food (5-dim mock embeddings)
docs = {
    "Python async tutorial":        np.array([0.9, 0.1, 0.2, 0.0, 0.1]),
    "RAG pipeline with ChromaDB":   np.array([0.8, 0.7, 0.3, 0.0, 0.1]),
    "LLM fine-tuning guide":        np.array([0.7, 0.5, 0.8, 0.0, 0.2]),
    "Transformer architecture":     np.array([0.85, 0.6, 0.7, 0.0, 0.0]),
    "How to make dosa at home":     np.array([0.0, 0.0, 0.1, 0.9, 0.8]),
    "Best biryani recipe":          np.array([0.0, 0.0, 0.0, 0.85, 0.9]),
    "Indian street food guide":     np.array([0.1, 0.0, 0.1, 0.8, 0.7]),
    "Deploying ML on GCP":          np.array([0.6, 0.3, 0.4, 0.0, 0.3]),
}

# Query: "how to make dosa"
query = np.array([0.05, 0.0, 0.1, 0.88, 0.75])

# TODO: compute similarity for each doc, sort, print

<details><summary>💡 Hint 1 — Conceptual</summary>

`np.dot(a, b)` gives the dot product. `np.linalg.norm(a)` gives the magnitude. Divide dot product by both magnitudes.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
results = [(cosine_similarity(query, v), name) for name, v in docs.items()]
for sim, doc in sorted(results, reverse=True):
    print(f'  {sim:.3f}  {doc}')
```

</details>

**✅ Success Criteria:**
- cosine_similarity function implemented correctly
- All 8 documents scored against the query
- Food documents rank in top 3
- Tech documents rank in bottom 4

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np

def cosine_similarity(a, b):
    """Cosine similarity: dot(a,b) / (norm(a) * norm(b))"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

docs = {
    "Python async tutorial":        np.array([0.9, 0.1, 0.2, 0.0, 0.1]),
    "RAG pipeline with ChromaDB":   np.array([0.8, 0.7, 0.3, 0.0, 0.1]),
    "LLM fine-tuning guide":        np.array([0.7, 0.5, 0.8, 0.0, 0.2]),
    "Transformer architecture":     np.array([0.85, 0.6, 0.7, 0.0, 0.0]),
    "How to make dosa at home":     np.array([0.0, 0.0, 0.1, 0.9, 0.8]),
    "Best biryani recipe":          np.array([0.0, 0.0, 0.0, 0.85, 0.9]),
    "Indian street food guide":     np.array([0.1, 0.0, 0.1, 0.8, 0.7]),
    "Deploying ML on GCP":          np.array([0.6, 0.3, 0.4, 0.0, 0.3]),
}

query = np.array([0.05, 0.0, 0.1, 0.88, 0.75])

results = [(cosine_similarity(query, v), name) for name, v in docs.items()]
for sim, doc in sorted(results, reverse=True):
    print(f'  {sim:.3f}  {doc}')

</details>

---

## Exercise 2 (Easy): Temperature Explorer

**📖 Concept:** Temperature divides logits before softmax: `softmax(logits / T)`. T→0 = argmax (always pick the top word). T→∞ = uniform (random). Production: T=0 for facts, T=0.7 for creativity, never T>1.2.

**📝 Task:**
1. Create 10 words with logits (raw model scores)
2. Apply softmax at temperatures: 0.01, 0.1, 0.5, 1.0, 3.0, 10.0
3. For each, print the top word, max probability, min probability
4. Find: at what temperature does the distribution become approximately uniform?

**📤 Expected Output:**
```
T=0.01  top=the   max=1.0000 min=0.0000 SPIKE
T=0.1   top=the   max=1.0000 min=0.0000 SPIKE
T=1.0   top=the   max=0.4029 min=0.0045 Peaked
T=10.0  top=the   max=0.1155 min=0.0827 ~UNIFORM
```

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np

def softmax(logits, temperature=1.0):
    """Softmax with temperature scaling."""
    scaled = np.array(logits) / max(temperature, 1e-8)
    exp = np.exp(scaled - np.max(scaled))  # Subtract max for numerical stability
    return exp / exp.sum()

words = ["the", "a", "is", "cat", "dog", "ran", "big", "small", "very", "not"]
logits = np.array([5.0, 3.5, 3.0, 4.2, 4.0, 2.5, 2.0, 1.5, 1.0, 0.5])

# TODO: loop through temperatures [0.01, 0.1, 0.5, 1.0, 3.0, 10.0]
# TODO: for each, print top word, max prob, min prob
# TODO: classify shape: SPIKE / Peaked / Spread / ~UNIFORM

<details><summary>💡 Hint 1 — Conceptual</summary>

At T=0.01, the highest logit (5.0) gets ~100% probability. At T=10.0, all words get ~10% each (1/10). The ratio of max/min probability tells you how "sharp" the distribution is.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
p = softmax(logits, temp)
ratio = p.max() / max(p.min(), 1e-10)
shape = '~UNIFORM' if ratio < 1.5 else 'Peaked' if ratio < 100 else 'SPIKE'
```

</details>

**✅ Success Criteria:**
- Softmax probabilities sum to 1.0 for all temperatures
- T=0.01 produces a spike (one word ~100%)
- T=10.0 produces near-uniform distribution
- Shape classification correctly identifies distribution type

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np

def softmax(logits, temperature=1.0):
    """Softmax with temperature scaling."""
    scaled = np.array(logits) / max(temperature, 1e-8)
    exp_vals = np.exp(scaled - np.max(scaled))
    return exp_vals / exp_vals.sum()

words = ["the", "a", "is", "cat", "dog", "ran", "big", "small", "very", "not"]
logits = np.array([5.0, 3.5, 3.0, 4.2, 4.0, 2.5, 2.0, 1.5, 1.0, 0.5])

for temp in [0.01, 0.1, 0.5, 1.0, 3.0, 10.0]:
    probs = softmax(logits, temp)
    ratio = probs.max() / max(probs.min(), 1e-10)
    if ratio < 1.5:
        shape = '~UNIFORM'
    elif ratio < 5:
        shape = 'Spread'
    elif ratio < 100:
        shape = 'Peaked'
    else:
        shape = 'SPIKE'
    top_word = words[np.argmax(probs)]
    print(f'T={temp:<5}  top={top_word:<5}  max={probs.max():.4f}  min={probs.min():.4f}  {shape}')

print('\nAt T≈10.0, distribution becomes approximately uniform (all words ~10%).')

</details>

---

## Exercise 3 (Medium): Attention from Scratch — Q/K/V

**📖 Concept:** Self-attention: `Attention(Q,K,V) = softmax(QK^T/√d_k) × V`. Q = "what am I looking for?", K = "what do I contain?", V = "what should I output?". Q, K, V are created by projecting input embeddings through learned weight matrices.

**📝 Task:**
1. Take 5 tokens with 8-dim embeddings
2. Project to Q, K, V (4-dim each) using random weight matrices
3. Compute attention scores: `Q @ K.T / sqrt(d_k)`
4. Apply softmax row-wise to get attention weights
5. Compute output: `attention_weights @ V`
6. Print which token attends most to which

**📤 Expected Output:**
```
Attention matrix:
          The     cat     sat      on     mat
  The   0.178   0.216*  0.196   0.197   0.213  → attends to 'cat'
  cat   0.191   0.189   0.213*  0.192   0.215  → attends to 'mat'
  ...
Output shape: (5, 4)
```

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
np.random.seed(42)

# 5 tokens, 8-dim embeddings
tokens = np.random.randn(5, 8) * 0.5
token_names = ["The", "cat", "sat", "on", "mat"]

# Weight matrices: project from 8D → 4D
W_q = np.random.randn(8, 4) * 0.1
W_k = np.random.randn(8, 4) * 0.1
W_v = np.random.randn(8, 4) * 0.1

# TODO: Q = tokens @ W_q, K = tokens @ W_k, V = tokens @ W_v
# TODO: scores = Q @ K.T / sqrt(d_k)
# TODO: attention_weights = softmax(scores) — apply row-wise!
# TODO: output = attention_weights @ V
# TODO: print attention matrix showing which token attends to which

<details><summary>💡 Hint 1 — Conceptual</summary>

`Q @ K.T` gives a (5,5) matrix: each token's "interest" in every other token. Division by √d_k prevents scores from getting too large (which would make softmax collapse to one-hot).

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# Row-wise softmax: each row sums to 1.0
attention = softmax(scores)
```

</details>

**✅ Success Criteria:**
- Q, K, V projected correctly: shape (5, 4) each
- Scores matrix shape: (5, 5)
- Softmax applied row-wise (each row sums to 1.0)
- Output shape: (5, 4)
- Each token's max-attention target identified

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)

tokens = np.random.randn(5, 8) * 0.5
token_names = ["The", "cat", "sat", "on", "mat"]

W_q = np.random.randn(8, 4) * 0.1
W_k = np.random.randn(8, 4) * 0.1
W_v = np.random.randn(8, 4) * 0.1

Q = tokens @ W_q
K = tokens @ W_k
V = tokens @ W_v

d_k = K.shape[1]
scores = (Q @ K.T) / np.sqrt(d_k)

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

attention = softmax(scores)

print('Attention matrix (who attends to whom):')
print(f'{"":>8}', end='')
for name in token_names:
    print(f'{name:>8}', end='')
print()
for i, name in enumerate(token_names):
    print(f'  {name:>6}', end='')
    max_j = np.argmax(attention[i])
    for j in range(5):
        marker = '*' if j == max_j else ' '
        print(f'  {attention[i, j]:.3f}{marker}', end='')
    print(f"  → attends to '{token_names[max_j]}'")

output = attention @ V
print(f'\nOutput shape: {output.shape} — context-aware representations!')
print(f'Row sums (should all be 1.0): {attention.sum(axis=1).round(3)}')

</details>

---

## Exercise 4 (Medium): Perplexity Calculator

**📖 Concept:** Perplexity = `exp(average cross-entropy loss)`. It answers: "on average, how many words is the model choosing between?" Perplexity 50,000 = random guessing. Perplexity 15 = decent. Perplexity ~6 = GPT-4 level. **Interview tip:** perplexity increasing during fine-tuning = catastrophic forgetting.

**📝 Task:**
1. Compute per-token cross-entropy: `-log(P(correct))`
2. Average across all tokens in a sequence
3. Perplexity = `exp(average_cross_entropy)`
4. Compare: random, bad, average, GPT-3, GPT-4 level
5. Print a table showing all model comparisons

**📤 Expected Output:**
```
Model                  Avg P(correct)   Avg CE   Perplexity
------------------------------------------------------------
  Random (50K vocab)          0.0000    10.820     50000.0
  Bad model                   0.0200     3.912        50.0
  Excellent (GPT-4)           0.8630     0.150         1.2
```

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np

def cross_entropy(prob_correct):
    """Per-token cross-entropy: -log(P(correct token))"""
    # TODO: return -np.log(prob + epsilon)
    pass

def perplexity(avg_ce):
    """Perplexity = exp(average cross-entropy)"""
    # TODO: return np.exp(avg_ce)
    pass

# Each list = P(correct token) for 10 tokens in a sequence
models = {
    "Random (50K vocab)": [1/50000] * 10,
    "Bad model":          [0.02] * 10,
    "Average model":      [0.15, 0.20, 0.10, 0.25, 0.12, 0.18, 0.22, 0.15, 0.20, 0.13],
    "Good (GPT-3)":       [0.55, 0.70, 0.45, 0.80, 0.60, 0.65, 0.75, 0.50, 0.70, 0.55],
    "Excellent (GPT-4)":  [0.85, 0.90, 0.78, 0.92, 0.88, 0.85, 0.91, 0.80, 0.88, 0.86],
}

# TODO: for each model, compute avg CE, then perplexity
# TODO: print table

<details><summary>💡 Hint 1 — Conceptual</summary>

Use `np.log()` (natural log) and `np.exp()`. Add a tiny epsilon (1e-10) inside the log to prevent log(0). Perplexity = "how many equally-likely words is the model confused between".

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
ces = [cross_entropy(p) for p in probs]
avg_ce = np.mean(ces)
ppl = perplexity(avg_ce)
print(f'{name}: perplexity = {ppl:.1f}')
```

</details>

**✅ Success Criteria:**
- Cross-entropy computed correctly for each token
- Perplexity = exp(avg cross-entropy)
- Random model has highest perplexity (~50,000)
- GPT-4 model has lowest perplexity
- Table format shows all model comparisons

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np

def cross_entropy(prob_correct):
    """Per-token cross-entropy: -log(P(correct token))"""
    return -np.log(prob_correct + 1e-10)

def perplexity(avg_ce):
    """Perplexity = exp(average cross-entropy)"""
    return np.exp(avg_ce)

models = {
    "Random (50K vocab)": [1/50000] * 10,
    "Bad model":          [0.02] * 10,
    "Average model":      [0.15, 0.20, 0.10, 0.25, 0.12, 0.18, 0.22, 0.15, 0.20, 0.13],
    "Good (GPT-3)":       [0.55, 0.70, 0.45, 0.80, 0.60, 0.65, 0.75, 0.50, 0.70, 0.55],
    "Excellent (GPT-4)":  [0.85, 0.90, 0.78, 0.92, 0.88, 0.85, 0.91, 0.80, 0.88, 0.86],
}

print(f'{"Model":<22} {"Avg P(correct)":>14} {"Avg CE":>8} {"Perplexity":>11}')
print('-' * 60)
for name, probs in models.items():
    ces = [cross_entropy(p) for p in probs]
    avg_ce = np.mean(ces)
    ppl = perplexity(avg_ce)
    print(f'  {name:<20} {np.mean(probs):>13.4f} {avg_ce:>8.3f} {ppl:>10.1f}')

print('\nLower perplexity = better model. GPT-4 ≈ 6 (chooses between ~6 words).')

</details>

---

## Exercise 5 (Hard): Backpropagation for 3-Layer Network

**📖 Concept:** Backpropagation uses the **chain rule** to trace error from output back through each layer. GELU activation (used in GPT, BERT, Gemini) is smoother than ReLU — it lets small negative values through, producing better gradients.

**📝 Task:**
1. Build Input(1) → Layer1(4) → GELU → Layer2(4) → GELU → Layer3(1)
2. Forward pass: compute output for input x=2.0, target y=8.0
3. Backward pass: chain rule for all 3 weight matrices
4. Train for 100 epochs
5. Compare learning rates 0.001, 0.01, 0.1 — which converges fastest?

**📤 Expected Output:**
```
lr=0.001:
  Epoch   0: loss=63.2041
  Final loss=45.7312   (too slow)

lr=0.01:
  Epoch   0: loss=63.2041
  Final loss=0.0001    (converges well!)

lr=0.1:
  Epoch   0: loss=63.2041
  Final loss=0.0000    (fastest)
```

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np

def gelu(x):
    """GELU activation (used in GPT, BERT, Gemini)."""
    return x * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def gelu_deriv(x):
    """Approximate GELU derivative."""
    cdf = 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))
    pdf = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)
    return cdf + x * pdf

x = np.array([[2.0]])
y_true = np.array([[8.0]])

for lr in [0.001, 0.01, 0.1]:
    np.random.seed(42)  # Same init for fair comparison
    # TODO: initialize w1(1,4), w2(4,4), w3(4,1)
    # TODO: train loop for 100 epochs:
    #   Forward: z1 → a1=gelu(z1) → z2 → a2=gelu(z2) → z3
    #   Loss: MSE
    #   Backward: chain rule
    #   Update: w -= lr * dw
    pass

<details><summary>💡 Hint 1 — Conceptual</summary>

Chain rule for w1: `∂L/∂w1 = ∂L/∂z3 × ∂z3/∂a2 × ∂a2/∂z2 × ∂z2/∂a1 × ∂a1/∂z1 × ∂z1/∂w1`. Each step is a matrix multiplication. The GELU derivative is needed at each activation layer.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
# Backward pass:
dz3 = 2 * (z3 - y_true)
dw3 = a2.T @ dz3
da2 = dz3 @ w3.T
dz2 = da2 * gelu_deriv(z2)
dw2 = a1.T @ dz2
da1 = dz2 @ w2.T
dz1 = da1 * gelu_deriv(z1)
dw1 = x.T @ dz1
```

</details>

**✅ Success Criteria:**
- Forward pass computes correct output shape
- Backward pass uses chain rule through all 3 layers
- GELU derivative applied at each activation layer
- Loss decreases over epochs
- lr=0.001 converges slowly, lr=0.01 and lr=0.1 converge well

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np

def gelu(x):
    return x * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def gelu_deriv(x):
    cdf = 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))
    pdf = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)
    return cdf + x * pdf

x = np.array([[2.0]])
y_true = np.array([[8.0]])

for lr in [0.001, 0.01, 0.1]:
    np.random.seed(42)
    w1 = np.random.randn(1, 4) * 0.5
    w2 = np.random.randn(4, 4) * 0.5
    w3 = np.random.randn(4, 1) * 0.5

    print(f'lr={lr}:')
    for epoch in range(100):
        # Forward
        z1 = x @ w1
        a1 = gelu(z1)
        z2 = a1 @ w2
        a2 = gelu(z2)
        z3 = a2 @ w3
        loss = float(((z3 - y_true) ** 2).mean())

        # Backward
        dz3 = 2 * (z3 - y_true)
        dw3 = a2.T @ dz3
        da2 = dz3 @ w3.T
        dz2 = da2 * gelu_deriv(z2)
        dw2 = a1.T @ dz2
        da1 = dz2 @ w2.T
        dz1 = da1 * gelu_deriv(z1)
        dw1 = x.T @ dz1

        # Update
        w3 -= lr * dw3
        w2 -= lr * dw2
        w1 -= lr * dw1

        if epoch % 25 == 0:
            print(f'  Epoch {epoch:>3}: loss={loss:.4f}')

    print(f'  Final loss={loss:.4f}\n')

</details>

---

## Exercise 6 (Hard): Mini Word2Vec from Scratch

**📖 Concept:** Word2Vec trains by predicting context words. Words in similar contexts (king/queen both near 'rules', 'crown') get similar vectors. The analogy `king − man + woman ≈ queen` emerges automatically from training. This is the foundation of all modern embeddings.

**📝 Task:**
1. Create 10 sentences with related word pairs (king/queen, man/woman, boy/girl)
2. Build vocabulary, initialize random 4D embeddings for each word
3. Train: push target and context word vectors closer together (200 epochs)
4. Test: king↔queen should be high similarity, king↔office should be low
5. Bonus: compute king − man + woman = ? (should be closest to queen)

**📤 Expected Output:**
```
  king↔queen:  0.912  (high — similar roles)
  man↔woman:   0.897  (high — similar contexts)
  king↔office: 0.234  (low — different contexts)

  king - man + woman = queen (0.876)
```

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
np.random.seed(42)

sentences = [
    ["king", "rules", "kingdom"],
    ["queen", "rules", "kingdom"],
    # TODO: 8 more sentences with related word pairs
]

# TODO: build vocab, init 4D embeddings
# TODO: train for 200 epochs (push words closer to context)
# TODO: test king↔queen (should be high), king↔office (should be low)
# TODO: analogy: king - man + woman = ?

<details><summary>💡 Hint 1 — Conceptual</summary>

The training loop: for each word in each sentence, nudge its embedding closer to its neighbor words (within a window of ±2). This makes co-occurring words get similar vectors over time.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
# Training loop:
for epoch in range(200):
    for sent in sentences:
        for i, target in enumerate(sent):
            for j in range(max(0, i-2), min(len(sent), i+3)):
                if i == j: continue
                diff = embeddings[w2i[sent[j]]] - embeddings[w2i[target]]
                embeddings[w2i[target]] += 0.005 * diff

# Analogy: king - man + woman = ?
result = embeddings[w2i['king']] - embeddings[w2i['man']] + embeddings[w2i['woman']]
```

</details>

**✅ Success Criteria:**
- Vocabulary built from all 10 sentences
- Embeddings trained for 200 epochs
- king↔queen similarity > 0.7
- king↔office similarity < 0.5
- Analogy king − man + woman ≈ queen

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)

sentences = [
    ["king", "rules", "kingdom"],
    ["queen", "rules", "kingdom"],
    ["king", "wears", "crown"],
    ["queen", "wears", "crown"],
    ["man", "works", "office"],
    ["woman", "works", "office"],
    ["prince", "is", "young", "king"],
    ["princess", "is", "young", "queen"],
    ["boy", "is", "young", "man"],
    ["girl", "is", "young", "woman"],
]

vocab = sorted(set(w for s in sentences for w in s))
w2i = {w: i for i, w in enumerate(vocab)}
print(f'Vocabulary ({len(vocab)} words): {vocab}\n')

embeddings = np.random.randn(len(vocab), 4) * 0.1

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

# Train
for epoch in range(200):
    for sent in sentences:
        for i, target in enumerate(sent):
            for j in range(max(0, i - 2), min(len(sent), i + 3)):
                if i == j:
                    continue
                diff = embeddings[w2i[sent[j]]] - embeddings[w2i[target]]
                embeddings[w2i[target]] += 0.005 * diff

# Test
print('Similarity tests:')
for w1, w2 in [("king", "queen"), ("man", "woman"), ("boy", "girl"), ("king", "office")]:
    sim = cosine_sim(embeddings[w2i[w1]], embeddings[w2i[w2]])
    print(f'  {w1}↔{w2}: {sim:.3f}')

print('\nAnalogy: king - man + woman = ?')
result_vec = embeddings[w2i['king']] - embeddings[w2i['man']] + embeddings[w2i['woman']]
exclude = {'king', 'man', 'woman'}
sims = sorted(
    [(cosine_sim(result_vec, embeddings[w2i[w]]), w) for w in vocab if w not in exclude],
    reverse=True
)
print(f'  Answer: {sims[0][1]} (similarity: {sims[0][0]:.3f})')
print(f'  Top 3: {[(w, f"{s:.3f}") for s, w in sims[:3]]}')

</details>

---

## 🎉 Done!

You now understand **all 9 math concepts** behind GenAI:
- **Vectors + Cosine Similarity** → RAG retrieval (Module 6)
- **Softmax + Temperature** → LLM generation control (every module)
- **Gradient Descent + Backpropagation** → How models learn (Module 2, 7)
- **Matrix Multiplication + Attention (Q/K/V)** → Core of transformers (Module 2)
- **Cross-Entropy + Perplexity** → Training signal + evaluation metric
- **GELU Activation** → Why depth matters in neural networks
- **Word2Vec** → Foundation of all modern embeddings

Coverage: 100% of what top GenAI courses teach. Next: **Module 1.**